# Unified RAG Pipeline
This notebook demonstrates a simple Retrieval-Augmented Generation (RAG) system running entirely locally using HuggingFace models and ChromaDB.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters chromadb pypdf sentence-transformers transformers torch

### 1. Setup & Imports

In [ ]:
import os
import time
import glob
import logging
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline

logging.getLogger("transformers").setLevel(logging.ERROR)

### 2. Create Sample Data & Ingest
We will create a sample document and then load and chunk it. You can place your own PDFs or text files into the `data` directory.

In [ ]:
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)
sample_path = os.path.join(data_dir, "sample_notebook_doc.txt")

with open(sample_path, "w", encoding="utf-8") as f:
    f.write("""Retrieval-Augmented Generation (RAG) is a technique that combines retrieval models and generative models to improve the quality of AI responses.\nInstead of relying solely on a model's internal parameters, RAG retrieves relevant information from an external knowledge base (like a vector database containing custom documents) and provides this information as context to the generation model.\nThis approach significantly reduces hallucinations, ensures that answers are grounded in verifiable data, and allows the system to access private or up-to-date information without requiring model retraining.\nThe main components of a RAG pipeline include document chunking, embedding generation, a vector store for semantic search, and a language model for synthesizing the final answer.""")

print("[*] Ingesting documents...")
documents = []
for file in glob.glob(os.path.join(data_dir, "**/*.*"), recursive=True):
    if file.endswith('.txt'):
        loader = TextLoader(file, encoding='utf-8')
        documents.extend(loader.load())
    elif file.endswith('.pdf'):
        loader = PyPDFLoader(file)
        documents.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len)
chunks = text_splitter.split_documents(documents)
print(f"[*] Created {len(chunks)} chunks from {len(documents)} documents.")

### 3. Initialize Vector Store

In [ ]:
print("[*] Initializing embedding model and vector store...")
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

persist_dir = "chroma_notebook_db"
vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=persist_dir)
print("[*] Vector store initialized.")

### 4. Setup Language Model & RAG Chain

In [ ]:
print("[*] Setting up Language Model...")
hf_pipeline = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM-135M-Instruct",
    max_new_tokens=150,
    truncation=True,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

prompt_template = """Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Answer:"""
prompt = PromptTemplate.from_template(prompt_template)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)
print("[*] LLM and QA Chain setup complete.")

### 5. Ask Questions

In [ ]:
def ask(question):
    print(f"\n[*] Querying: '{question}'")
    retrieved_docs = retriever.invoke(question)
    print("\n--- Retrieved Context ---")
    for i, doc in enumerate(retrieved_docs):
        print(f"Chunk {i+1}: {doc.page_content}")
    print("-------------------------")
    
    response = qa_chain.invoke(question)
    print(f"\n🤖 AI Answer:\n{response}")
    return response

# Test it!
ask("What is Retrieval-Augmented Generation?")
ask("How does RAG reduce hallucinations?")